In [19]:
import re
import itertools
from itertools import chain
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import polars as pl

import statsmodels.formula.api as smf
import statsmodels.stats.anova as sa
from statsmodels.stats.multitest import fdrcorrection

from scipy.stats import mannwhitneyu

from novami.io.file import read_pl, write_pl

# ChEMBL Deposition

In [81]:
df = read_pl("../data/ChEMBL/processed/chembl_measurements.xlsx")

## Mechanism / Readout missingness

In [82]:
frac_missing = df.filter(
    pl.any_horizontal(
        pl.col("assay_readout") == "AMB",
        pl.col("assay_mechanism") == "AMB",
        )
    )["assay_chembl_id"].n_unique() / df["assay_chembl_id"].n_unique()

print(f"Percent of not-annotated mechanisms/readouts: {frac_missing:.2%}")

Percent of not-annotated mechanisms/readouts: 33.26%


## ChEMBL deposition by year

In [58]:
df = read_pl("../data/ChEMBL/processed/chembl_measurements.xlsx")

In [63]:
total_counts = df.group_by("target_channel_name").agg(pl.col("molregno").n_unique().alias("Compounds"))

In [64]:
df = df.rename({"year":"Year"}).filter(pl.col("Year").is_not_null())

In [65]:
ddc = defaultdict(list)

min_year = df["Year"].min()
max_year = df["Year"].max()

year_range = list(np.arange(min_year, max_year + 1))
ddc["Year"] = year_range

for channel in df["target_channel_name"].unique().to_list():
    sub_df = df.filter(
        pl.col("target_channel_name") == channel
    )
    for year in year_range:
        year_df = sub_df.filter(
            pl.col("Year") <= year
        )
        n_unique = year_df["molregno"].n_unique()
        ddc[channel].append(
            n_unique
        )

In [66]:
count_df = pl.DataFrame(ddc)

In [67]:
df_long = count_df.unpivot(
    on=["Nav1.5", "hERG", "Kir2.1", "Cav1.2", "Kv7.1", "Kv4.3"],
    index="Year",
    variable_name="target_channel_name",
    value_name="Compounds"
)

In [70]:
write_pl(df_long, "../data/deposition/by_year.xlsx")
write_pl(total_counts, "../data/deposition/total.xlsx")

# Model and descriptor usage counts

In [146]:
study_df = read_pl('../data/supporting_tables/s1.xlsx')

In [147]:
time_breaks = [(2001, 2009), (2010, 2018), (2019, 2026)]

In [148]:
# Check the most frequently used descriptors

for period in time_breaks:
    print(f'Descriptors between {period[0]} and {period[1]}:')
    sub_df = study_df.filter(pl.col('Year').is_between(*period))
    dscs = sub_df['Features'].to_list()
    dscs_types = [re.sub(r'\([^)]*\)', '', item) for item in dscs]
    dscs_types = chain.from_iterable([item.split(',') for item in dscs_types])
    dscs_types = [item.strip() for item in dscs_types]

    ct = Counter(dscs_types)
    print('\t', ct.most_common(10))

Descriptors between 2001 and 2009:
	 [('MD', 19), ('3D Shape-based', 6), ('Fragment', 4), ('Pharmacophore', 3), ('Circular', 3), ('QC', 3), ('Field-based', 3), ('Fragment-based', 2), ('Topological', 2), ('Atom Types', 2)]
Descriptors between 2010 and 2018:
	 [('MD', 21), ('Circular', 13), ('Fragment', 10), ('Pharmacophore', 7), ('Path-based', 6), ('Field-based', 3), ('Topological', 3), ('4D-FP', 2), ('Atom Pairs', 2), ('custom SMARTS', 1)]
Descriptors between 2019 and 2026:
	 [('MD', 44), ('Circular', 40), ('Fragment', 29), ('Graphs', 22), ('Path-based', 11), ('Atom Pairs', 7), ('Embeddings', 6), ('Pharmacophore', 6), ('SMILES', 6), ('Embedding', 5)]


In [149]:
# Check the most frequently used models

for period in time_breaks:
    print(f'Models between {period[0]} and {period[1]}:')
    sub_df = study_df.filter(pl.col('Year').is_between(*period))
    models = sub_df['Models'].to_list()
    models = [re.sub(r'\([^)]*\)', '', item) for item in models]
    models = chain.from_iterable([item.split(',') for item in models])
    models = [item.strip() for item in models]
    ct = Counter(models)
    print('\t', ct.most_common(10))

Models between 2001 and 2009:
	 [('PLS', 9), ('SVM', 8), ('DT', 6), ('NB', 4), ('Alignment-based', 3), ('SOM', 3), ('MLR', 3), ('LR', 3), ('kNN', 3), ('NN', 2)]
Models between 2010 and 2018:
	 [('SVM', 15), ('RF', 14), ('PLS', 10), ('kNN', 8), ('NB', 5), ('DT', 4), ('LDA', 3), ('GBM', 3), ('GP', 2), ('MLR', 2)]
Models between 2019 and 2026:
	 [('RF', 37), ('SVM', 26), ('XGB', 18), ('kNN', 14), ('NB', 10), ('MLP', 10), ('DNN', 9), ('> DNN', 8), ('GCN', 7), ('GBM', 7)]


# Performance analysis

## Prepare relevant tables

In [191]:
df_cls = read_pl("../data/supporting_tables/s3.xlsx")
df_reg = read_pl("../data/supporting_tables/s4.xlsx")

In [192]:
df_cls = df_cls.with_columns(
    pl.col("Splitting/Evaluation").map_elements(
        lambda evl: re.sub(r"\\cite\{[\w\s,]+\}", "", evl).strip(),
        return_dtype=pl.String
    ).alias("EVL")
)

df_reg = df_reg.with_columns(
    pl.col("Splitting/Evaluation").map_elements(
        lambda evl: re.sub(r"\\cite\{[\w\s,]+\}", "", evl).strip(),
        return_dtype=pl.String
    ).alias("EVL")
)

In [193]:
all_evaluations = set(df_cls["EVL"].drop_nulls().unique().to_list()).union(set(df_reg["EVL"].drop_nulls().unique().to_list()))

In [194]:
evl_map = {
    'D-Optimal': "Uniform",
    'DISE': "Distance",
    'Distance': "Distance",
    'Diverse': "Uniform",
    'Diverse train' : "Uniform",
    'Ext' : "External",
    'Ext (Temporal)': "External",
    'External': "External",
    'In-house': None,
    'LCO CV': "LOO",
    'LOO CV': "LOO",
    'Manual': "Manual",
    'N/A': None,
    'Random' : "Random",
    'Random CV' :"Random",
    'Random CV: binding': "Random",
    'Random CV: clamp': "Random",
    'Random Nested CV': "Random",
    'Random: all assays': "Random",
    'Random: patch-clamp': "Random",
    'Scaffold': "Scaffold",
    'Scaffold CV': "Scaffold",
    'Scaffold Distance' : "Distance",
    'Sorted Uniform' : "Uniform",
    'Sorted Y-based': "Uniform",
    'Stratified CV' : "Stratified",
    'Stratified Random': "Stratified",
    'Stratified Random CV': "Stratified",
    'Temporal': "Temporal",
    None: None
}

In [195]:
df_cls = df_cls.with_columns(
    pl.col("EVL").map_elements(
        lambda evl: evl_map.get(evl, None),
        return_dtype=pl.String
    ).alias("Eval Class")
)

df_reg = df_reg.with_columns(
    pl.col("EVL").map_elements(
        lambda evl: evl_map.get(evl, None),
        return_dtype=pl.String
    ).alias("Eval Class")
)

In [196]:
df_cls = df_cls.with_columns(
    pl.col("Model").map_elements(
        lambda mod: re.search(r"([^\s]+)", mod).group(1),
        return_dtype=pl.String
    ).alias("Model Class")
)

df_reg = df_reg.with_columns(
    pl.col("Model").map_elements(
        lambda mod: re.search(r"([^\s]+)", mod).group(1),
        return_dtype=pl.String
    ).alias("Model Class")
)

In [199]:
write_pl(df_cls, "../data/performance/classification.xlsx")
write_pl(df_reg, "../data/performance/regression.xlsx")

## ANOVA analysis

In [200]:
class_df = read_pl("../data/performance/classification.xlsx")
reg_df = read_pl("../data/performance/regression.xlsx")

cdf = class_df.filter(
    pl.col("Threshold").is_in([
        "<10,>30", "10", "1", "<1,>10"
    ]),
    pl.col("Model Class").is_in([
        "ML", "DL"
    ]),
    pl.col("Eval Class").is_in([
        "Random", "External", "Distance"
    ]),
    pl.col("Channel") == "hERG"
)

rdf = reg_df.filter(
    pl.col("Model Class").is_in([
        "DL", "ML"
    ]),
    pl.col("Eval Class").is_in([
        "External", "Random", "Distance"
    ]),
    pl.col("Channel") == "hERG"
)

cdf = cdf.select(["Threshold", "ACC", "BA", "SEN", "SPE", "ROC AUC", "MCC", "Year", "Eval Class", "Model Class"])

rdf = rdf.select(["R2", "MAE", "RMSE", "Year", "Eval Class", "Model Class"])

cdf = cdf.unpivot(
    index=["Threshold", "Year", "Eval Class", "Model Class"],
    variable_name="Metric",
    value_name="Value",
)

rdf = rdf.unpivot(
    index=["Year", "Eval Class", "Model Class"],
    variable_name="Metric",
    value_name="Value",
)

cdf = cdf.filter(
    pl.col("Value").is_not_null()
)
rdf = rdf.filter(
    pl.col("Value").is_not_null()
)

cdf = cdf.rename({"Eval Class": "Eval", "Model Class": "Model"})
rdf = rdf.rename({"Eval Class": "Eval", "Model Class": "Model"})

In [201]:
cdf = cdf.with_columns(
    pl.when(
        pl.col("Year") <= 2009
    ).then(
        pl.lit("Foundation")
    ).when(
        pl.col("Year") <= 2018
    ).then(
        pl.lit("Consolidation")
    ).otherwise(
        pl.lit("Expansion")
    ).alias("Era")
)

rdf = rdf.with_columns(
    pl.when(
        pl.col("Year") <= 2009
    ).then(
        pl.lit("Foundation")
    ).when(
        pl.col("Year") <= 2018
    ).then(
        pl.lit("Consolidation")
    ).otherwise(
        pl.lit("Expansion")
    ).alias("Era")
)

In [202]:
write_pl(cdf, "../results/anova/cdf.xlsx")
write_pl(rdf, "../results/anova/rdf.xlsx")

In [203]:
cdf = cdf.to_pandas()

results = []

for metric in cdf["Metric"].unique():
    sub_df = cdf[cdf["Metric"] == metric]

    model = smf.ols(
        "Value ~ C(Model) + C(Eval) + C(Threshold) + C(Era)",
        data=sub_df
    ).fit()

    table = sa.anova_lm(model, typ=2)
    table["eta_squared"] = table["sum_sq"] / table["sum_sq"].sum()
    table.index = ["Model", "Eval", "Threshold", "Era", "Residual"]
    table.insert(0, "Metric", metric)
    results.append(table)

cres = pd.concat(results).sort_values(["Metric", "eta_squared"], ascending=[True, False])

In [204]:
rdf = rdf.to_pandas()

results = []

for metric in rdf["Metric"].unique():
    sub_df = rdf[rdf["Metric"] == metric]

    model = smf.ols(
        "Value ~ C(Model) + C(Eval) + C(Era)",
        data=sub_df
    ).fit()

    table = sa.anova_lm(model, typ=2)
    table["eta_squared"] = table["sum_sq"] / table["sum_sq"].sum()
    table.index = ["Model", "Eval", "Era", "Residual"]
    table.insert(0, "Metric", metric)
    results.append(table)

rres = pd.concat(results).sort_values(["Metric", "eta_squared"], ascending=[True, False])

In [205]:
write_pl(pl.from_pandas(cres), "../results/anova/classification.xlsx")
write_pl(pl.from_pandas(rres), "../results/anova/regression.xlsx")

## Mann-Whitney U test

In [207]:
cdf = read_pl("../results/anova/cdf.xlsx")
rdf = read_pl("../results/anova/rdf.xlsx")

In [209]:
def rank_biserial(u_stat: float, n1: int, n2: int):
    """
    Rank biserial effect size for Mann-Whitney U-test. Positive value correspond to values
    from population 1 being larger than from population 2.

    Parameters
    ----------
    u_stat : float
        Test statistic from scipy.stats.mannwhitneyu
    n1: int
        Size of population 1.
    n2: int
        Size of population 2.
    """

    return (2 * u_stat) / (n1 * n2) - 1

In [210]:
def round_to_significant(x: float, n: int):
    """
    Round a value to a given number of significant digits.
    """
    if isinstance(x, str):
        return(x)
    if x is None:
        return 'None'

    if x == 0:
        return 0
    else:
        return round(x, n - int(np.floor(np.log10(abs(x)))) - 1)

In [211]:
def mwu_comparison(values_1: np.ndarray, values_2: np.ndarray, label_1: str, label_2: str, metric: str,
                   comparison: str, min_n: int = 3, alpha: float = 0.05):
    """
    Run a Mann-Whitney U test.

    Parameters
    ----------
    values_1: np.ndarray
        Samples from population 1
    values_2: np.ndarray
        Samples from population 2
    label_1: str
        Name for population 1
    label_2: str
        Name for population 2
    metric: str
        Performance metric to evaluated
    comparison: str
        Evaluation column
    min_n: int = 3
        Minimum number of samples to consider output valid
    alpha: float
        Significance level
    """

    null_hypothesis = f"{label_1} ~ {label_2}"
    n1, n2 = values_1.shape[0], values_2.shape[0]
    if any([n1 < 3, n2 < 3]):
        print(f"Sample size too small for {comparison}: <{null_hypothesis}>.\nN1: {n1}, N2: {n2}")
        return {
            "Metric": metric,
            "Evaluation": comparison,
            "Null Hypothesis": null_hypothesis,
            "Label1": label_1,
            "Label2": label_2,
            "N1": n1,
            "N2": n2,
            "Mean1": None,
            "Mean2": None,
            "Median1": None,
            "Median2": None,
            "U_stat": None,
            "p_value": None,
            "rbc": None,
            "effect_size": None,
            "low_n_flag": True,
            "outcome": None
    }
    stat, p = mannwhitneyu(
        x=values_1,
        y=values_2,
        alternative="two-sided"
    )

    rb = rank_biserial(stat, n1, n2)
    arb = np.abs(rb)

    if arb <= 0.1:
        effect_size = "Negligible"
    elif arb <= 0.3:
        effect_size = "Small"
    elif arb <= 0.5:
        effect_size = "Medium"
    else:
        effect_size = "Large"

    if p < alpha:
        if rb > 0:
            sig = ">"
        elif rb == 0:
            sig = "~"
        else:
            sig = "<"
    else:
        sig = "~"

    outcome = f"{label_1} {sig} {label_2}"

    return {
        "Metric": metric,
        "Evaluation": comparison,
        "Null Hypothesis": null_hypothesis,
        "Label1": label_1,
        "Label2": label_2,
        "N1": n1,
        "N2": n2,
        "Mean1": round_to_significant(values_1.mean(), 5),
        "Mean2": round_to_significant(values_2.mean(), 5),
        "Median1": round_to_significant(np.median(values_1), 5),
        "Median2": round_to_significant(np.median(values_2), 5),
        "U_stat": round_to_significant(stat, 5),
        "p_value": round_to_significant(p, 5),
        "rbc": round_to_significant(rb, 5),
        "effect_size": effect_size,
        "low_n_flag": any([n1 < min_n, n2 < min_n]),
        "outcome": outcome
    }

In [212]:
pairs_cls = [
    ("Threshold", [*itertools.combinations(cdf["Threshold"].unique().to_list(), r=2)]),
    ("Eval", [*itertools.combinations(cdf["Eval"].unique().to_list(), r=2)]),
    ("Model", [*itertools.combinations(cdf["Model"].unique().to_list(), r=2)]),
    ("Era", [*itertools.combinations(cdf["Era"].unique().to_list(), r=2)])
]

pairs_reg = [
    ("Eval", [*itertools.combinations(rdf["Eval"].unique().to_list(), r=2)]),
    ("Model", [*itertools.combinations(rdf["Model"].unique().to_list(), r=2)]),
    ("Era", [*itertools.combinations(rdf["Era"].unique().to_list(), r=2)])
]

In [213]:
def run_analysis(df: pl.DataFrame, pairs: list, min_n: int = 3, alpha: float = 0.05) -> pl.DataFrame:
    results = []
    for metric in df["Metric"].unique():
        mdf = df.filter(pl.col("Metric") == metric)
        rows = []
        for column, group_pairs in pairs:
            for label_1, label_2 in group_pairs:
                values_1 = mdf.filter(pl.col(column) == label_1)["Value"].to_numpy()
                values_2 = mdf.filter(pl.col(column) == label_2)["Value"].to_numpy()
                out = mwu_comparison(
                    values_1=values_1, values_2=values_2,
                    label_1=label_1, label_2=label_2,
                    metric=metric, comparison=column,
                    min_n=min_n, alpha=alpha
                )
                rows.append(out)

        metric_df = pl.DataFrame(rows)
        valid = metric_df.filter(pl.col("p_value").is_not_null())
        invalid = metric_df.filter(pl.col("p_value").is_null())

        if len(valid) > 0:
            _, p_adj = fdrcorrection(valid["p_value"].to_numpy(), alpha=alpha)
            valid = valid.with_columns(pl.Series("p_value_adj", p_adj, dtype=pl.Float64))

        metric_df = pl.concat([valid, invalid], how="diagonal_relaxed")
        results.append(metric_df)

    combined = pl.concat(results, how="diagonal_relaxed")
    combined = combined.with_columns(
        pl.struct(["Label1", "Label2", "rbc", "p_value_adj"])
        .map_elements(
            lambda r: (
                f"{r['Label1']} > {r['Label2']}" if r["rbc"] > 0
                else f"{r['Label1']} < {r['Label2']}"
            ) if (r["p_value_adj"] is not None and r["p_value_adj"] < alpha)
            else (
                f"{r['Label1']} ~ {r['Label2']}" if r["p_value_adj"] is not None
                else None
            ),
            return_dtype=pl.String
        )
        .alias("outcome_adj")
    ).sort(["Metric", "p_value_adj"], nulls_last=True)

    return combined


In [214]:
cls_df = run_analysis(cdf, pairs_cls)
reg_df = run_analysis(rdf, pairs_reg)

Sample size too small for Era: <Foundation ~ Consolidation>.
N1: 0, N2: 9
Sample size too small for Era: <Foundation ~ Expansion>.
N1: 0, N2: 62
Sample size too small for Threshold: <<1,>10 ~ 1>.
N1: 2, N2: 6
Sample size too small for Threshold: <<1,>10 ~ <10,>30>.
N1: 2, N2: 3
Sample size too small for Threshold: <<1,>10 ~ 10>.
N1: 2, N2: 60
Sample size too small for Era: <Foundation ~ Consolidation>.
N1: 2, N2: 8
Sample size too small for Era: <Foundation ~ Expansion>.
N1: 2, N2: 61
Sample size too small for Threshold: <<1,>10 ~ 1>.
N1: 1, N2: 0
Sample size too small for Threshold: <<1,>10 ~ <10,>30>.
N1: 1, N2: 0
Sample size too small for Threshold: <<1,>10 ~ 10>.
N1: 1, N2: 19
Sample size too small for Threshold: <1 ~ <10,>30>.
N1: 0, N2: 0
Sample size too small for Threshold: <1 ~ 10>.
N1: 0, N2: 19
Sample size too small for Threshold: <<10,>30 ~ 10>.
N1: 0, N2: 19
Sample size too small for Eval: <External ~ Distance>.
N1: 8, N2: 2
Sample size too small for Eval: <Distance ~ Rando

In [215]:
write_pl(cls_df, "../results/mwu/classification.xlsx")
write_pl(reg_df, "../results/mwu/regression.xlsx")

In [216]:
# Output as LaTeX tables

In [217]:
cls_df = cls_df.drop(["Null Hypothesis", "Mean1", "Mean2", "U_stat", "outcome"])
reg_df = reg_df.drop(["Null Hypothesis", "Mean1", "Mean2", "U_stat", "outcome"])

In [218]:
cls_df = cls_df.cast({
    "N1": pl.Int64,
    "N2": pl.Int64,
    "Median1": pl.Float64,
    "Median2": pl.Float64,
    "rbc": pl.Float64,
    "p_value_adj": pl.Float64,
    "p_value": pl.Float64,
})

reg_df = reg_df.cast({
    "N1": pl.Int64,
    "N2": pl.Int64,
    "Median1": pl.Float64,
    "Median2": pl.Float64,
    "rbc": pl.Float64,
    "p_value_adj": pl.Float64,
    "p_value": pl.Float64,
})

In [222]:
def print_table(df: pl.DataFrame, task="cls"):

    if task == "cls":
        metrics = ["ACC", "BA", "SEN", "SPE", "ROC AUC", "MCC"]
        evals = ["Era", "Model", "Eval", "Threshold"]
    else:
        metrics = ["R2", "RMSE", "MAE"]
        evals = ["Era", "Model", "Eval"]

    for idx, ev in enumerate(evals):
        if idx != 0:
            print("\t" + r"\midrule")
        print("\t" + r"\multicolumn{12}{c}{" + f"{ev}" + "}" + r" \\")
        print("\t" + r"\midrule")
        edf = df.filter(pl.col("Evaluation") == ev).drop("Evaluation").sort(["Metric", "p_value_adj", "p_value"], nulls_last=True)

        for idx, mt in enumerate(metrics):
            mdf = edf.filter(pl.col("Metric") == mt)
            mt_line = "\t" + r"\multirow{" + f"{len(mdf)}" + "}{*}{" + f"{mt}" + "}"
            print(mt_line)

            mx_lab = np.max([len(item) for item in (mdf["Label1"].to_list() + mdf["Label2"].to_list())])
            mx_n = np.max([len(str(item)) for item in (mdf["N1"].to_list() + mdf["N2"].to_list())])
            mx_m = np.max([len(str(round_to_significant(item, 2))) for item in (mdf["Median1"].to_list() + mdf["Median2"].to_list())])

            for row in mdf.iter_rows(named=True):
                g1, g2 = row["Label1"], row["Label2"]
                n1, n2 = row["N1"], row["N2"]
                m1, m2 = row["Median1"], row["Median2"]
                pv, pva = row["p_value"], row["p_value_adj"]
                if pv is not None:
                    pv = round_to_significant(pv, 2)
                    pva = round_to_significant(pva, 2)
                    m1 = round_to_significant(m1, 2)
                    m2 = round_to_significant(m2, 2)
                    outcome = row["outcome_adj"].replace("~", r"$\sim$")
                rbc, e_size, low_flag = row["rbc"], row["effect_size"], row["low_n_flag"]

                if pv is None:
                    row_line = "\t\t" + f"& {g1:<{mx_lab}} & {n1:<{mx_n}} & - " + " "*3 + f"& {g2:<{mx_lab}} & {n2:<{mx_n}} & - " + " "*3   + f"& - & - "            + f"& - & - " + f"& -" + r" \\"
                else:
                    row_line = "\t\t" + f"& {g1:<{mx_lab}} & {n1:<{mx_n}} & {m1:.2f} " + f"& {g2:<{mx_lab}} & {n2:<{mx_n}} & {m2:.2f} " + f"& {rbc:.2f} & {e_size} " + f"& {pv} & {pva} " + f"& {outcome}" + r" \\"
                print(row_line)
            if idx != len(metrics) -1:
                print("\t" + r"\cmidrule{2-12}")

In [223]:
print_table(cls_df, task="cls")

	\multicolumn{12}{c}{Era} \\
	\midrule
	\multirow{3}{*}{ACC}
		& Foundation    & 6  & 0.90 & Expansion     & 77 & 0.83 & 0.42 & Medium & 0.086 & 0.22 & Foundation $\sim$ Expansion \\
		& Foundation    & 6  & 0.90 & Consolidation & 17 & 0.85 & 0.32 & Medium & 0.26 & 0.52 & Foundation $\sim$ Consolidation \\
		& Consolidation & 17 & 0.85 & Expansion     & 77 & 0.83 & 0.06 & Negligible & 0.72 & 0.78 & Consolidation $\sim$ Expansion \\
	\cmidrule{2-12}
	\multirow{3}{*}{BA}
		& Foundation    & 0  & -    & Consolidation & 2  & -    & - & - & - & - & - \\
		& Foundation    & 0  & -    & Expansion     & 18 & -    & - & - & - & - & - \\
		& Consolidation & 2  & -    & Expansion     & 18 & -    & - & - & - & - & - \\
	\cmidrule{2-12}
	\multirow{3}{*}{SEN}
		& Foundation    & 8  & 0.86 & Expansion     & 75 & 0.83 & 0.15 & Small & 0.49 & 0.9 & Foundation $\sim$ Expansion \\
		& Foundation    & 8  & 0.86 & Consolidation & 16 & 0.85 & 0.13 & Small & 0.62 & 0.9 & Foundation $\sim$ Consolidation \\
		

In [224]:
print_table(reg_df, task="reg")

	\multicolumn{12}{c}{Era} \\
	\midrule
	\multirow{3}{*}{R2}
		& Foundation    & 5  & 0.85 & Expansion     & 16 & 0.62 & 0.65 & Large & 0.035 & 0.25 & Foundation $\sim$ Expansion \\
		& Foundation    & 5  & 0.85 & Consolidation & 3  & 0.68 & 0.47 & Medium & 0.37 & 0.87 & Foundation $\sim$ Consolidation \\
		& Consolidation & 3  & 0.68 & Expansion     & 16 & 0.62 & 0.31 & Medium & 0.43 & 0.87 & Consolidation $\sim$ Expansion \\
	\cmidrule{2-12}
	\multirow{3}{*}{RMSE}
		& Foundation    & 3  & 0.74 & Expansion     & 19 & 0.60 & 0.42 & Medium & 0.27 & 0.45 & Foundation $\sim$ Expansion \\
		& Foundation    & 3  & -    & Consolidation & 1  & -    & - & - & - & - & - \\
		& Consolidation & 1  & -    & Expansion     & 19 & -    & - & - & - & - & - \\
	\cmidrule{2-12}
	\multirow{3}{*}{MAE}
		& Foundation    & 2  & -    & Consolidation & 2  & -    & - & - & - & - & - \\
		& Foundation    & 2  & -    & Expansion     & 10 & -    & - & - & - & - & - \\
		& Consolidation & 2  & -    & Expansion     

# Other statistics